# PIANO AMT (Onsets & Frames) — Simple Public Demo

This notebook is a **simplified public Colab demo** for your PyTorch Onsets & Frames project.
It keeps the flow closer to the Magenta-style demo notebook:

1. clone repo and install requirements  
2. download public checkpoint + demo samples from Hugging Face  
3. load model and print transparency info  
4. choose **one public demo sample** or **upload custom audio**  
5. run transcription once  
6. show results, MIDI, evaluation, and downloads  

### What this version removes
- no live threshold sliders  
- no complex widget controls  
- no private Google Drive dependency  

### What this version keeps
- your public Hugging Face checkpoint and demo assets  
- your helper layer in `demo/`  
- your evaluation tables and failure analysis for public demo samples  

This version assumes your **public GitHub repo is flat**, with files directly at repo root:

- `requirements_demo.txt`
- `demo/`
- `demo_assets/`
- `notebooks/`

In [ ]:
#@title 0. Public demo configuration
REPO_URL = "https://github.com/Mobinmo83/AMT.git"
REPO_BRANCH = "main"
REPO_CLONE_DIR = "/content/repo"

HF_REPO_ID_OVERRIDE = "Mobinmo83/piano-amt-demo"
HF_CHECKPOINT_FILENAME_OVERRIDE = "checkpoints/best.pt"
MODEL_COMPLEXITY_OVERRIDE = 48

In [ ]:
#@title 1. Clone repo and install public demo requirements
import os
import shutil
from pathlib import Path

repo_dir = Path(REPO_CLONE_DIR)
if repo_dir.exists():
    shutil.rmtree(repo_dir)

!git clone -q --branch "$REPO_BRANCH" "$REPO_URL" "$REPO_CLONE_DIR"
%cd $REPO_CLONE_DIR
!pip -q install -r requirements_demo.txt huggingface_hub

In [ ]:
#@title 2. Imports and runtime setup
import json
import os
import sys
import uuid
import shutil
from pathlib import Path

import pandas as pd
import torch
from IPython.display import Audio, FileLink, Markdown, display
from huggingface_hub import hf_hub_download

PROJECT_ROOT = REPO_CLONE_DIR

os.environ["AMT_DEMO_HF_REPO_ID"] = HF_REPO_ID_OVERRIDE
os.environ["AMT_DEMO_HF_CHECKPOINT_FILENAME"] = HF_CHECKPOINT_FILENAME_OVERRIDE
os.environ["AMT_DEMO_MODEL_COMPLEXITY"] = str(MODEL_COMPLEXITY_OVERRIDE)
os.environ["AMT_DEMO_REPO_ROOT"] = PROJECT_ROOT

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from demo.checkpoint import load_demo_model, format_checkpoint_summary, maybe_torchinfo_summary
from demo.demo_config import (
    DEFAULT_FRAME_THRESHOLD,
    DEFAULT_ONSET_THRESHOLD,
    HTML_DIR,
    MIDI_DIR,
    PLOT_DIR,
    ensure_demo_dirs,
)
from demo.evaluation import evaluate_prediction_vs_gt, failure_table, main_scores_table
from demo.inference import decode_prediction_to_pretty_midi, run_model_on_mel, save_prediction_midi
from demo.rendering import (
    plot_piano_roll_side_by_side,
    plot_roll_diff,
    render_bokeh_midi,
    synthesize_pretty_midi,
)
from demo.sources import (
    audio_path_to_mel,
    list_demo_sample_names,
    load_ground_truth_labels,
    resolve_demo_sample_paths,
    save_uploaded_audio,
)

ensure_demo_dirs()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
#@title 3. Download public demo assets from Hugging Face
PROJECT_ROOT_PATH = Path(PROJECT_ROOT)
LOCAL_MANIFEST = PROJECT_ROOT_PATH / "demo" / "sample_manifest.json"

manifest_cache_path = hf_hub_download(
    repo_id=HF_REPO_ID_OVERRIDE,
    filename="demo/sample_manifest.json",
)
LOCAL_MANIFEST.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(manifest_cache_path, LOCAL_MANIFEST)

with open(LOCAL_MANIFEST, "r", encoding="utf-8") as f:
    manifest = json.load(f)

for sample in manifest["samples"]:
    for key in ["audio", "labels"]:
        remote_path = sample[key]
        cached_file = hf_hub_download(
            repo_id=HF_REPO_ID_OVERRIDE,
            filename=remote_path,
        )
        local_target = PROJECT_ROOT_PATH / remote_path
        local_target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(cached_file, local_target)

print("Downloaded public demo assets:")
for p in sorted((PROJECT_ROOT_PATH / "demo_assets").glob("*")):
    print(" -", p)

In [ ]:

#@title 4. Load checkpoint and show transparency info
MODEL, CKPT, CKPT_PATH = load_demo_model(DEVICE)

print(format_checkpoint_summary(MODEL, CKPT, CKPT_PATH))
print("
Compact torchinfo summary:
")
print(maybe_torchinfo_summary(MODEL))
print("
Full architecture:
")
print(MODEL)


In [ ]:
#@title 5. List public demo samples
DEMO_SAMPLE_NAMES = list_demo_sample_names()
print("Available public demo samples:\n")
for i, name in enumerate(DEMO_SAMPLE_NAMES):
    print(f"[{i}] {name}")

In [ ]:

#@title 6. Choose source
SOURCE_MODE = "Public demo sample" #@param ["Public demo sample", "Upload custom audio"]
DEMO_SAMPLE_INDEX = 0 #@param {type:"integer"}

print("Source mode:", SOURCE_MODE)
if SOURCE_MODE == "Public demo sample":
    if not DEMO_SAMPLE_NAMES:
        raise RuntimeError("No public demo samples were found.")
    if DEMO_SAMPLE_INDEX < 0 or DEMO_SAMPLE_INDEX >= len(DEMO_SAMPLE_NAMES):
        raise IndexError(f"DEMO_SAMPLE_INDEX must be between 0 and {len(DEMO_SAMPLE_NAMES)-1}")
    print("Selected sample:", DEMO_SAMPLE_NAMES[DEMO_SAMPLE_INDEX])
else:
    print("You will upload an audio file in the next cell.")


In [ ]:

#@title 7. Prepare input audio
from pathlib import Path

if SOURCE_MODE == "Public demo sample":
    selected_name = DEMO_SAMPLE_NAMES[DEMO_SAMPLE_INDEX]
    audio_path, labels_path = resolve_demo_sample_paths(selected_name, PROJECT_ROOT)
    mel = audio_path_to_mel(audio_path)
    gt = load_ground_truth_labels(labels_path)
    title = Path(audio_path).stem
    source_type = "demo_sample"
    print("Prepared public demo sample:", title)
else:
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("Please upload an audio file.")
    first_name = next(iter(uploaded.keys()))
    audio_path = save_uploaded_audio(uploaded[first_name], first_name)
    mel = audio_path_to_mel(audio_path)
    gt = None
    title = Path(audio_path).stem
    source_type = "custom_upload"
    print("Prepared uploaded audio:", title)

print("Audio path:", audio_path)
print("Mel shape:", tuple(mel.shape))


In [ ]:

#@title 8. Run transcription
pred = run_model_on_mel(MODEL, mel, DEVICE)
print("Prediction tensor shapes:")
for k, v in pred.items():
    print(f"  {k}: {tuple(v.shape)}")


In [ ]:

#@title 9. Decode to MIDI and save outputs
run_id = f"{title}_{uuid.uuid4().hex[:8]}"
pred_midi_path = MIDI_DIR / f"{run_id}_pred.mid"
pred_png_path = PLOT_DIR / f"{run_id}_pred_roll.png"
diff_png_path = PLOT_DIR / f"{run_id}_diff.png"
pred_html_path = HTML_DIR / f"{run_id}_pred_plot.html"

pred_pm = decode_prediction_to_pretty_midi(pred, DEFAULT_ONSET_THRESHOLD, DEFAULT_FRAME_THRESHOLD)
pred_audio = synthesize_pretty_midi(pred_pm)
save_prediction_midi(pred, pred_midi_path, DEFAULT_ONSET_THRESHOLD, DEFAULT_FRAME_THRESHOLD)

print("Saved predicted MIDI:", pred_midi_path)


In [ ]:

#@title 10. Show prediction results

display(Markdown(f"## Results — {title}"))
display(Markdown(f"**Source:** `{source_type}`"))
display(Markdown("### Original audio"))
display(Audio(filename=str(audio_path)))

display(Markdown("### Predicted MIDI audio"))
display(Audio(pred_audio, rate=16000))

display(Markdown("### Predicted interactive MIDI"))
plot = render_bokeh_midi(pred_pm, html_path=pred_html_path, show_inline=True)
if plot is None:
    print("Interactive MIDI plot unavailable; HTML export skipped.")

display(Markdown("### Predicted piano roll"))
fig = plot_piano_roll_side_by_side(
    pred_frame=pred["frame"],
    gt_frame=None,
    title=f"Prediction — {title}",
    save_path=pred_png_path,
)
display(fig)


In [ ]:

#@title 11. Show ground truth and evaluation (public demo samples only)
if gt is None:
    display(Markdown("> Uploaded custom audio has no ground-truth labels, so evaluation and original MIDI comparison are skipped."
    ))
else:
    gt_midi_path = MIDI_DIR / f"{run_id}_ground_truth.mid"
    gt_html_path = HTML_DIR / f"{run_id}_ground_truth_plot.html"
    gt_png_path = PLOT_DIR / f"{run_id}_ground_truth_roll.png"

    gt_pm = decode_prediction_to_pretty_midi(gt, 0.5, 0.5)
    gt_audio = synthesize_pretty_midi(gt_pm)
    save_prediction_midi(gt, gt_midi_path, 0.5, 0.5)

    display(Markdown("### Ground-truth MIDI audio"))
    display(Audio(gt_audio, rate=16000))

    display(Markdown("### Ground-truth interactive MIDI"))
    gt_plot = render_bokeh_midi(gt_pm, html_path=gt_html_path, show_inline=True)
    if gt_plot is None:
        print("Interactive ground-truth MIDI plot unavailable.")

    display(Markdown("### Predicted vs ground-truth piano rolls"))
    comp_fig = plot_piano_roll_side_by_side(
        pred_frame=pred["frame"],
        gt_frame=gt["frame"],
        title=title,
        save_path=gt_png_path,
    )
    display(comp_fig)

    display(Markdown("### Diff plot"))
    diff_fig = plot_roll_diff(
        pred_frame=pred["frame"],
        gt_frame=gt["frame"],
        frame_threshold=DEFAULT_FRAME_THRESHOLD,
        save_path=diff_png_path,
    )
    display(diff_fig)

    results = evaluate_prediction_vs_gt(pred, gt, DEFAULT_ONSET_THRESHOLD, DEFAULT_FRAME_THRESHOLD)

    display(Markdown("### Quantitative evaluation"))
    display(main_scores_table(results).style.format({"Value": "{:.4f}"}))

    display(Markdown("### Failure-case analysis"))
    display(failure_table(results).style.format({"Value": "{:.4f}"}))


In [ ]:

#@title 12. Download links

display(Markdown("### Download files"))
print("Predicted MIDI:")
display(FileLink(str(pred_midi_path)))
print("Predicted piano-roll PNG:")
display(FileLink(str(pred_png_path)))
if pred_html_path.exists():
    print("Predicted interactive MIDI HTML:")
    display(FileLink(str(pred_html_path)))

if gt is not None:
    print("Ground-truth MIDI:")
    display(FileLink(str(gt_midi_path)))
    if gt_html_path.exists():
        print("Ground-truth interactive MIDI HTML:")
        display(FileLink(str(gt_html_path)))
    print("Diff plot PNG:")
    display(FileLink(str(diff_png_path)))
